# TianWen — one-click validation

Validate the whole **VLM → detector fusion** pipeline online, with no local setup.
Runs on a free CPU; if a GPU is present it is used automatically.

What this notebook proves, end to end:
1. the test suite passes (real YOLO + real CLIP + Lightning, in CI too),
2. the full stack **learns** (single-batch overfit through real fusion),
3. you can **ship just the detector** (export a VLM-free checkpoint and run inference).

> First run downloads model weights (YOLOv8n ~6 MB, CLIP ~600 MB) — a minute or two.

## 1. Setup — clone & install

In [ ]:
import os
REPO = "https://github.com/Hollis36/TianWen.git"
# New code (CLIP VLM, synthetic data, export, ...) is on PR #16's branch until it
# merges; this tries the branch and falls back to main once the branch is gone.
BRANCH = "claude/upbeat-hypatia-0pjsV"
if not os.path.exists('TianWen'):
    if os.system(f'git clone --depth 1 -b {BRANCH} {REPO}') != 0:
        os.system(f'git clone --depth 1 {REPO}')  # branch merged/deleted -> use main
%cd TianWen
!pip -q install -e ".[dev]"

## 2. Correctness — run the test suite

The same suite GitHub Actions runs on every push (real models, real training, exported detector).

In [ ]:
# Fast subset that needs no large downloads; drop '-k' to run everything.
!python -m pytest tests/test_decision_fusion.py tests/test_synthetic_dataset.py -q

## 3. The stack really trains

Real YOLOv8n + a real CLIP teacher + feature fusion. We overfit one fixed batch and watch the
loss drop — proof the losses are real and gradients flow through the fusion (not the old zero placeholder).

In [ ]:
import torch
from tianwen.detectors.yolo import YOLODetector
from tianwen.vlms.clip_vlm import CLIPVLM
from tianwen.fusions.feature_fusion import FeatureFusion

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

detector = YOLODetector(model_name='yolov8n', num_classes=5, input_size=(320, 320), pretrained=True).to(device)
vlm = CLIPVLM(model_name='openai/clip-vit-base-patch32').to(device)   # real CLIP teacher
fusion = FeatureFusion(detector=detector, vlm=vlm, fusion_level='neck').to(device)
fusion.train()

torch.manual_seed(0)
images = torch.rand(2, 3, 320, 320, device=device)
targets = [
    {'boxes': torch.tensor([[40., 40, 200, 200], [20., 20, 90, 90]], device=device), 'labels': torch.tensor([0, 3], device=device)},
    {'boxes': torch.tensor([[60., 60, 260, 260]], device=device), 'labels': torch.tensor([2], device=device)},
]
opt = torch.optim.AdamW([p for p in fusion.parameters() if p.requires_grad], lr=1e-3)
losses = []
for step in range(25):
    opt.zero_grad()
    loss = sum(fusion(images, targets).loss_dict.values())
    loss.backward(); opt.step()
    losses.append(float(loss.detach()))
print(f'loss: {losses[0]:.2f} -> {losses[-1]:.2f}')
assert losses[-1] < losses[0] * 0.7, 'expected the loss to drop'
print('OK — the full detector + CLIP + fusion stack is learning.')

## 4. Ship just the detector

Export a standalone detector (no VLM weights or dependencies) and run inference with it — the deploy artifact.

In [ ]:
from tianwen.utils.export import export_detector_checkpoint, load_detector_checkpoint

cfg = {'type': 'yolov8', 'model_name': 'yolov8n', 'num_classes': 5, 'input_size': (320, 320)}
payload = export_detector_checkpoint(fusion.detector, cfg, 'detector.pt', class_names=[f'class_{i}' for i in range(5)])
has_vlm = any('clip' in k.lower() or 'vlm' in k for k in payload['state_dict'])
print('detector.pt size:', round(os.path.getsize('detector.pt') / 1e6, 1), 'MB | contains VLM weights:', has_vlm)

deployed = load_detector_checkpoint('detector.pt', map_location=device).to(device)
deployed.eval()
with torch.no_grad():
    out = deployed(torch.rand(1, 3, 320, 320, device=device))
print('standalone (VLM-free) inference OK — images:', len(out.outputs))

## 5. Zero-data CPU smoke via the CLI (optional)

Drive the whole pipeline through the Hydra CLI on synthetic data — no dataset needed.

In [ ]:
!python tools/train.py dataset=dummy detector=yolov8 detector.model_name=yolov8n \
    vlm=clip fusion=feature_fusion trainer.fast_dev_run=true trainer.accelerator=cpu

## Next steps

- Real training + first benchmark on a free GPU (Kaggle has COCO ready-made): see
  [`docs/ONLINE_VALIDATION.md`](https://github.com/Hollis36/TianWen/blob/main/docs/ONLINE_VALIDATION.md).
- Train on COCO: `python tools/train.py detector=yolov8 vlm=clip fusion=distillation dataset=coco`
- Export & evaluate: `python tools/export.py -c runs/.../last.ckpt -o detector.pt`